In [0]:
CREATE SCHEMA IF NOT EXISTS workspace.gold;

-- One row per project, with KPIs and risk flag
CREATE OR REPLACE TABLE workspace.gold.project_kpis AS
WITH base AS (
  SELECT *,
         ROUND(spend_usd / NULLIF(budget_usd, 0), 4)                    AS spend_pct,
         budget_usd - orig_budget_usd                                    AS budget_growth_usd,
         ROUND((budget_usd - orig_budget_usd) / NULLIF(orig_budget_usd, 0), 4) AS budget_growth_pct,
         DATEDIFF(fcst_end_date, orig_end_date)                          AS delay_days
  FROM workspace.silver.projects
)
SELECT *,
       CASE WHEN COALESCE(budget_growth_pct, 0) > 0.20 OR COALESCE(delay_days, 0) > 365 THEN 'Red'
            WHEN COALESCE(budget_growth_pct, 0) > 0.10 OR COALESCE(delay_days, 0) > 180 THEN 'Amber'
            ELSE 'Green' END AS risk_status
FROM base;

-- Agency summary
CREATE OR REPLACE TABLE workspace.gold.agency_summary AS
SELECT agency,
       COUNT(*)                                        AS projects,
       ROUND(SUM(budget_usd) / 1e6, 1)                 AS budget_m,
       ROUND(SUM(spend_usd) / 1e6, 1)                  AS spend_m,
       ROUND(SUM(spend_usd) / NULLIF(SUM(budget_usd), 0), 3) AS spend_pct,
       ROUND(AVG(budget_growth_pct), 3)                AS avg_budget_growth_pct,
       ROUND(AVG(delay_days), 0)                       AS avg_delay_days,
       SUM(CASE WHEN risk_status = 'Red' THEN 1 ELSE 0 END) AS red_projects
FROM workspace.gold.project_kpis
GROUP BY agency;

-- Monthly trend from spend history snapshots
CREATE OR REPLACE TABLE workspace.gold.monthly_trend AS
SELECT month,
       COUNT(DISTINCT fms_id)                         AS projects,
       ROUND(SUM(budget_usd) / 1e6, 1)                AS total_budget_m,
       ROUND(SUM(spend_usd) / 1e6, 1)                 AS total_spend_m,
       ROUND(SUM(budget_variance_usd) / 1e6, 2)       AS budget_variance_m
FROM workspace.silver.spend_history
GROUP BY month
ORDER BY month;
-- Quick look
SELECT risk_status, COUNT(*) AS projects, ROUND(SUM(budget_usd) / 1e9, 2) AS budget_billion
FROM workspace.gold.project_kpis GROUP BY risk_status;